# BTZSC-RU — прогон в Google Colab

Ноутбук **вызывает** `experiment.py` из клонированного репозитория.
Код проекта загружается из <https://github.com/clinch0/ML2> (ячейка 2).
Откройте .ipynb в Colab и выполняйте ячейки сверху вниз.

**Токен Hugging Face не нужен**: все модели и датасеты публичные.

Runtime → Change runtime type → **T4 GPU**. Без GPU энкодеры считаются на CPU, LLM получают `SKIPPED_NO_GPU`.

**Ничего выбирать не нужно.** `MODE = "all"` в ячейке 1 прогоняет всё по порядку:
preflight → **anchor** (полные сплиты) → block_a (300) → block_b (300) → export →
finetune / extra_models / ru_extension → export_final.
Достаточно нажать Runtime → Run all. Отдельные режимы остались на случай, когда нужен один этап.

## Что делает прогон: два блока моделей

| Блок | Модели | Данные | Зачем |
|---|---|---|---|
| **А** | 5 моделей из статьи: all-MiniLM-L6-v2, e5-base-v2, e5-large-v2, bge-base-en-v1.5, bge-large-en-v1.5 | 7 датасетов статьи по 300 примеров | воспроизвести её результат нашим кодом |
| **Б** | 5 наших (FRIDA, USER-bge-m3, rubert-mini-frida, multilingual-e5-base, Vikhr-Qwen-2.5-1.5B) + 2 кросс-энкодера: bge-reranker-v2-m3 (568M) и rubert-base-cased-nli-threeway (178M) | те же данные, та же выборка | сравнить с блоком А на одной шкале; реранкер и NLI закрывают семейства 3 и 4 из пяти в статье |

Якорь к публикации включает IMDb и AG News на полном сплите (бэкенд st): проверка гипотезы
об обрезке длины у e5 — это +18–22 минуты. Кросс-энкодеры в блоке Б — ещё ~15 минут
(44 400 пар на модель). Расширения (дообучение, остальные модели, русские датасеты,
увеличенные выборки) включаются флагами в ячейке 1 и по умолчанию выключены.

Порядок этапов подобран так, чтобы ценное считалось первым: сначала основная оценка и экспорт,
потом тяжёлые этапы на больших выборках. Если сессия Colab умрёт на тяжёлом этапе, архив с
основными результатами уже сохранён, а повторный запуск с тем же `RUN_ID` досчитает остальное
(resume). Скачайте архив результатов до завершения сессии Colab.

Ориентировочное время на T4: основная часть с длинным якорем и кросс-энкодерами ~80 минут,
дополнительные этапы ещё ~60 минут при
`FULL_CAP = 3000`. Общий дедлайн `DEADLINE_MIN` не даёт прогону уйти в бесконечность.

In [1]:
# 1. Параметры прогона
MODE = "all"                # ВСЁ САМО: preflight → anchor → БЛОК А → БЛОК Б → export.
                            # Менять не нужно. Этапы по отдельности:
                            # preflight | smoke | evaluate | replicate | finetune | export
N_SAMPLES = None            # None = 300 примеров на датасет в режиме all; число = вручную
RUN_ID = "run1"             # тот же RUN_ID = продолжение прогона (resume)
# Код проекта клонируется из репозитория:
# Только адрес репозитория: ссылку /tree/... на страницу ветки git clone не принимает.
GIT_REPO_URL = "https://github.com/clinch0/ML2.git"
GIT_BRANCH = "feature/anchor-cross-encoders"
PROJECT_DIR = "/content/btzsc_project"
HF_CACHE = "/content/hf_cache"
RUNS_ROOT = "/content/runs"
# Ничего отключать не нужно: по умолчанию выполняется весь конвейер.
# Флаги ниже — аварийные, если времени в сессии совсем нет.
# По умолчанию считается всё, на чём держатся выводы отчёта:
# якорь → блок А → блок Б → дообучение → русское расширение.
SKIP_FINETUNE = False       # выключить дообучение rubert-tiny2 (иначе +1-3 мин, даёт слайд «разметка против размера»)
SKIP_RU_EXTENSION = False   # выключить русские датасеты и разрыв EN−RU (иначе +20 мин)
WITH_EXTRA_MODELS = True    # остальные модели сверх блока Б: tiny-энкодеры и две LLM (+30 мин)
WITH_FULL_TESTS = False     # увеличенные выборки для блоков А и Б (+60 мин)
# Якорь к публикации (полные сплиты пяти датасетов, включая IMDb и AG News) включён всегда:
# это единственные числа прогона, сравнимые с Таблицей 2 статьи. Отключать не рекомендуется.
SKIP_ANCHOR = False
DEADLINE_MIN = 150          # общий дедлайн: тяжёлые этапы за ним помечаются NOT_RUN,
                            # повторный запуск с тем же RUN_ID досчитает их через resume
FULL_CAP = 3000             # потолок примеров на датасет для «полных» этапов (0 = весь сплит)
print("MODE:", MODE, "| RUN_ID:", RUN_ID, "| N_SAMPLES:", N_SAMPLES, "| репозиторий:", GIT_REPO_URL)

MODE: all | RUN_ID: run1 | N_SAMPLES: None | репозиторий: https://github.com/clinch0/ML2.git


In [2]:
# 2. Загрузка кода проекта
import os, shutil, subprocess

shutil.rmtree(PROJECT_DIR, ignore_errors=True)
repo_url = GIT_REPO_URL.rstrip("/").split("/tree/", 1)[0]
if repo_url != GIT_REPO_URL:
    print("Адрес страницы ветки преобразован в адрес репозитория:", repo_url)
cmd = ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, repo_url, PROJECT_DIR]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode:
    error = result.stderr.strip() or f"git clone завершился с кодом {result.returncode}"
    raise RuntimeError(f"Не удалось клонировать ветку {GIT_BRANCH} из {repo_url}: {error}")
if not os.path.isfile(os.path.join(PROJECT_DIR, "experiment.py")):
    raise FileNotFoundError("В клонированном репозитории нет experiment.py")
print("код проекта загружен:", PROJECT_DIR)
print(sorted(os.listdir(PROJECT_DIR))[:20])

код проекта загружен: /content/btzsc_project
['.git', '.github', '.gitignore', 'BTZSC_Colab.ipynb', 'README.md', 'btzsc_ru', 'contracts.json', 'data', 'dist', 'docs', 'experiment.py', 'figures', 'incoming', 'paper_scores', 'requirements-colab.txt', 'results', 'scripts', 'tests']


In [3]:
# 3. Установка зависимостей
!pip -q install -r {PROJECT_DIR}/requirements-colab.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.6 MB/s eta 0:00:00


In [4]:
# 4. Каталоги прогона и переменные окружения HF (до импорта huggingface_hub)
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # иначе лог тонет в «Loading weights»
os.environ["HF_HUB_DISABLE_XET"] = "1"            # и в «downloading bytes …» от xet-клиента
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DATASETS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HOME"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE, "datasets")
os.makedirs(HF_CACHE, exist_ok=True)

os.makedirs(RUNS_ROOT, exist_ok=True)
RUN_DIR = os.path.join(RUNS_ROOT, RUN_ID)
print("результаты прогона (локально):", RUN_DIR)

результаты прогона (локально): /content/runs/run1


In [5]:
# 5. Публичные модели и датасеты загружаются анонимно.
print("Hugging Face: анонимная загрузка публичных моделей и датасетов")

Hugging Face: анонимная загрузка публичных моделей и датасетов


In [6]:
# 6. Импорт experiment.py и план прогона (ноутбук вызывает модуль, а не копирует логику)
import sys

sys.path.insert(0, PROJECT_DIR)
import experiment

print("experiment:", experiment.__doc__.splitlines()[0])
print("режимы:", experiment.MODES)
%cd {PROJECT_DIR}
!python experiment.py --mode {MODE} --run-id {RUN_ID} --dry-run

experiment: Точка входа эксперимента BTZSC-RU.
режимы: ('all', 'preflight', 'block_a', 'block_b', 'smoke', 'baseline', 'evaluate', 'reference', 'ours_full', 'replicate', 'st_check', 'finetune', 'export')
/content/btzsc_project
mode=all run_id=run1 config_hash=690b7685339a
этапы: preflight → anchor → block_a → block_b → export → finetune → ru_extension → export_final
блок А: 5 моделей статьи, блок Б: 7 наших; 7 датасетов × 300 примеров = 10,500 предсказаний на блок
n_samples=300
datasets: btzsc_agnews[btzsc/btzsc:agnews/test], btzsc_imdb[btzsc/btzsc:imdb/test], btzsc_rottentomatoes[btzsc/btzsc:rottentomatoes/test], btzsc_financialphrasebank[btzsc/btzsc:financialphrasebank/test], btzsc_emotiondair[btzsc/btzsc:emotiondair/test], btzsc_banking77[btzsc/btzsc:banking77/test], btzsc_massive[btzsc/btzsc:massive/test], massive_scenario_en[mteb/amazon_massive_scenario:en/test], massive_scenario_ru[mteb/amazon_massive_scenario:ru/test], go_emotions_en[seara/ru_go_emotions:simplified/test], go_emo

In [7]:
# 7. Preflight: контракты датасетов и среда. Модели не загружаются.
%cd {PROJECT_DIR}
!python experiment.py --mode preflight --run-id {RUN_ID} --out {RUNS_ROOT} --cache-dir {HF_CACHE}
import os
print(open(os.path.join(RUN_DIR, "preflight.json"), encoding="utf-8").read()[:1500])

/content/btzsc_project
Generating test split: 100% 30400/30400 [00:00<00:00, 585352.92 examples/s]
Generating test split: 100% 20000/20000 [00:00<00:00, 291819.29 examples/s]
Generating test split: 100% 2132/2132 [00:00<00:00, 507823.05 examples/s]
Generating test split: 100% 2070/2070 [00:00<00:00, 476363.95 examples/s]
Generating test split: 100% 12000/12000 [00:00<00:00, 1137078.62 examples/s]
Generating test split: 100% 175466/175466 [00:00<00:00, 1817628.42 examples/s]
Generating test split: 100% 221760/221760 [00:00<00:00, 1773785.66 examples/s]
Generating train split: 100% 189891/189891 [00:06<00:00, 28631.91 examples/s]
Generating validation split: 100% 21098/21098 [00:00<00:00, 30999.88 examples/s]
Resolving data files: 100% 51/51 [00:00<00:00, 23228.31it/s]
Resolving data files: 100% 51/51 [00:00<00:00, 18863.27it/s]
Resolving data files: 100% 51/51 [00:00<00:00, 23070.48it/s]
Generating train split: 11514 examples [00:00, 634677.11 examples/s]
Generating test split: 2974 exa

In [8]:
# 8. Запуск. Основной запуск эксперимента. Этапы идут сами,
# Прогресс печатается структурированными строками: возврат каретки в Colab через поток не виден,
# поэтому бар рисуется текстом и печатается отдельными строками:
#   [block_b 2/5 · BERTA] btzsc_imdb 3/7 [######....] 60% · 180/300 · 14 мс/пример · фон: frida
#   где: этап и номер модели · текущий датасет · бар пары · сколько примеров · скорость ·
#        какая модель в это время качается в фоне
# В конце каждой пары — строка с macro-F1 и общим прогрессом этапа с оценкой остатка.
# с resume и записью после каждого батча:
#   preflight → anchor → block_a → block_b → export → finetune → extra_models → ru_extension → export_final
# Три теста прогона: anchor (модели статьи на полном сплите — проверка кода),
# block_a + block_b (обе группы моделей на одной выборке), ru_extension (русские задачи).
import shlex, subprocess

cmd = f"python experiment.py --mode {MODE} --run-id {RUN_ID} --out {RUNS_ROOT} --cache-dir {HF_CACHE}"
cmd += f" --deadline-min {DEADLINE_MIN} --full-cap {FULL_CAP}"
if N_SAMPLES is not None:
    cmd += f" --n-samples {N_SAMPLES}"
for flag, option in (
    (SKIP_ANCHOR, "--no-anchor"),
    (SKIP_FINETUNE, "--no-finetune"),
    (SKIP_RU_EXTENSION, "--no-ru-extension"),
    (WITH_EXTRA_MODELS, "--with-extra-models"),
    (WITH_FULL_TESTS, "--with-full-baseline --with-full-ours"),
):
    if flag:
        cmd += " " + option
print(cmd)

proc = subprocess.Popen(
    shlex.split(cmd), cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:          # лог идёт в реальном времени, а не одной простынёй в конце
    print(line, end="")
proc.wait()
print("код возврата:", proc.returncode)

import json
report_path = os.path.join(RUN_DIR, "run_all_report.json")
if os.path.exists(report_path):
    report = json.load(open(report_path, encoding="utf-8"))
    print()
    print("итог по этапам:")
    for stage, info in report.items():
        mark = "OK " if info["status"] == "OK" else "СБОЙ"
        print(f"  {mark} {stage:11} {info['seconds']:>7.1f} с")
        if info["status"] != "OK":
            print(f"       причина: {str(info['detail'])[:300]}")


python experiment.py --mode all --run-id run1 --out /content/runs --cache-dir /content/hf_cache --deadline-min 150 --full-cap 3000 --with-extra-models
  манифест прогона: 14 датасетов × 300 примеров
[preflight] OK за 25.7 с
  осталось времени по дедлайну: 150 мин
манифест пересоздан: выборка была на 300 примеров, запрошено весь сплит. Прежние строки results.csv остаются в истории.

=== этап baseline: 5 моделей × 5 датасетов × весь сплит примеров = 25 пар ===
[baseline 1/5 · all-MiniLM-L6-v2] загрузка весов… · фон: e5-base-v2
[baseline 1/5 · all-MiniLM-L6-v2] модель готова за 28 с · фон: e5-base-v2
[baseline 1/5 · all-MiniLM-L6-v2] btzsc_rottentomatoes 1/5 · 1066 примеров
[baseline 1/5 · all-MiniLM-L6-v2] btzsc_rottentomatoes 1/5 [██░░░░░░░░]  21% · 224/1066 · 7 мс/пример
[baseline 1/5 · all-MiniLM-L6-v2] btzsc_rottentomatoes 1/5 [████░░░░░░]  42% · 448/1066 · 4 мс/пример
[baseline 1/5 · all-MiniLM-L6-v2] btzsc_rottentomatoes 1/5 [██████░░░░]  63% · 672/1066 · 3 мс/пример
[baseline 1/5 

In [9]:
# 9. Быстрый просмотр результатов
import os, pandas as pd

res = os.path.join(RUN_DIR, "results.csv")
if os.path.exists(res):
    df = pd.read_csv(res)
    display(df[["model_id", "dataset_key", "macro_f1", "accuracy", "status", "reason"]])
    print(df["status"].value_counts().to_dict())
else:
    print("results.csv ещё нет — это нормально после preflight; для чисел нужен MODE='smoke' или 'evaluate'")

rep = os.path.join(RUN_DIR, "replication", "results.csv")
if os.path.exists(rep):
    ref = {
        ("sentence-transformers/all-MiniLM-L6-v2", "btzsc_agnews"): 0.495,
        ("sentence-transformers/all-MiniLM-L6-v2", "btzsc_imdb"): 0.340,
        ("intfloat/e5-base-v2", "btzsc_agnews"): 0.761,
        ("intfloat/e5-base-v2", "btzsc_imdb"): 0.898,
        ("intfloat/e5-large-v2", "btzsc_agnews"): 0.787,
        ("intfloat/e5-large-v2", "btzsc_imdb"): 0.928,
        ("BAAI/bge-base-en-v1.5", "btzsc_agnews"): 0.635,
        ("BAAI/bge-base-en-v1.5", "btzsc_imdb"): 0.896,
        ("BAAI/bge-large-en-v1.5", "btzsc_agnews"): 0.766,
        ("BAAI/bge-large-en-v1.5", "btzsc_imdb"): 0.935,
    }
    rdf = pd.read_csv(rep)
    rdf = rdf[rdf["status"] == "OK"].copy()
    rdf["статья"] = [ref.get((m, d)) for m, d in zip(rdf["model_id"], rdf["dataset_key"])]
    rdf["Δ"] = (rdf["macro_f1"] - rdf["статья"]).round(3)
    display(rdf[["model_id", "dataset_key", "macro_f1", "статья", "Δ", "n_samples"]])
    print("сверка с числами статьи: |Δ| max =", rdf["Δ"].abs().max())
else:
    print("сверки ещё нет — она появится после этапа replicate")

,model_id,dataset_key,macro_f1,accuracy,status,reason
0,sentence-transformers/all-MiniLM-L6-v2,btzsc_rottentomatoes,0.658970,0.659475,OK,NaN
1,sentence-transformers/all-MiniLM-L6-v2,btzsc_financialphrasebank,0.504811,0.514493,OK,NaN
2,sentence-transformers/all-MiniLM-L6-v2,btzsc_emotiondair,0.348269,0.375000,OK,NaN
3,sentence-transformers/all-MiniLM-L6-v2,btzsc_agnews,0.692926,0.702368,OK,NaN
4,sentence-transformers/all-MiniLM-L6-v2,btzsc_imdb,0.647765,0.667500,OK,NaN
...,...,...,...,...,...,...
265,Qwen/Qwen2.5-1.5B-Instruct,massive_ru,0.079087,0.120000,OK,NaN
266,Qwen/Qwen2.5-1.5B-Instruct,massive_scenario_en,0.530465,0.523333,OK,NaN
267,Qwen/Qwen2.5-1.5B-Instruct,massive_scenario_ru,0.491220,0.486667,OK,NaN
268,Qwen/Qwen2.5-1.5B-Instruct,go_emotions_en,0.085972,0.130000,OK,NaN


{'OK': 270}
сверки ещё нет — она появится после этапа replicate


In [10]:
# 10. Скачать архив. В режиме all экспорт уже выполнен ячейкой 8.
import os, subprocess, zipfile

zip_path = os.path.join(RUNS_ROOT, f"colab_outputs_{RUN_ID}.zip")
if not os.path.exists(zip_path):
    res = os.path.join(RUN_DIR, "results.csv")
    if not os.path.exists(res):
        print(f"В {RUN_DIR} нет results.csv — экспортировать нечего.")
        raise SystemExit("Сначала выполните ячейку 8.")
    proc = subprocess.run(
        ["python", "experiment.py", "--mode", "export", "--run-id", RUN_ID, "--out", RUNS_ROOT],
        capture_output=True, text=True, cwd=PROJECT_DIR,
    )
    print(proc.stdout[-2000:])

names = zipfile.ZipFile(zip_path).namelist() if os.path.exists(zip_path) else []
print(zip_path, os.path.getsize(zip_path) if names else 0, "байт | файлов:", len(names))
print(names)
assert names, "архив пуст — не скачивайте его, сначала выполните ячейку 11"

from google.colab import files
files.download(zip_path)

/content/runs/colab_outputs_run1.zip 854491 байт | файлов: 11
['results.csv', 'summary.csv', 'predictions.jsonl', 'sample_manifest.csv', 'manifest_info.json', 'run_config.json', 'environment.json', 'preflight.json', 'errors.log', 'ft_metrics.csv', 'ft_config.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>